# Problem 4:


In [1]:
import numpy as np
from scipy.io import loadmat
from scipy.optimize import minimize
from sklearn.preprocessing import OneHotEncoder

np.random.seed(301)


In [2]:
from pathlib import Path

data_path = Path('data/ex3data1.mat')
if not data_path.exists():
    raise FileNotFoundError('Expected dataset at data/ex3data1.mat')

data = loadmat(data_path)
X = data['X']
y = data['y']

print('X shape:', X.shape)
print('y shape:', y.shape)


X shape: (5000, 400)
y shape: (5000, 1)


In [3]:
def one_hot_encode(y):
    try:
        encoder = OneHotEncoder(sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(sparse=False)
    return encoder.fit_transform(y), encoder

y_onehot, encoder = one_hot_encode(y)

input_size  = 400
hidden_size1 = 20   # units in first hidden layer
hidden_size2 = 20   # units in second hidden layer
num_labels  = 10
learning_rate = 1.0

print('y_onehot shape:', y_onehot.shape)


y_onehot shape: (5000, 10)


In [4]:
def sigmoid(z):
    # Scaled sigmoid: σ̂(z) = 1 / (1 + e^{-2z}) — used in hidden layers only
    return 1.0 / (1.0 + np.exp(-2.0 * z))

def sigmoid_std(z):
    # Standard sigmoid: σ(z) = 1 / (1 + e^{-z}) — used in the output layer
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_gradient(z):
    # Derivative of scaled sigmoid: σ̂'(z) = 2 * σ̂(z) * (1 - σ̂(z))
    s = sigmoid(z)
    return 2.0 * s * (1.0 - s)

In [5]:
def forward_propagate(X, theta1, theta2, theta3):
    """3-layer forward pass: input → hidden1 (scaled sigmoid) → hidden2 (scaled sigmoid) → output (standard sigmoid)."""
    m = X.shape[0]
    # Layer 1 → 2 (scaled sigmoid activation)
    a1 = np.concatenate([np.ones((m, 1)), X], axis=1)          # m × (input+1)
    z2 = a1 @ theta1.T                                           # m × hidden1
    a2 = np.concatenate([np.ones((m, 1)), sigmoid(z2)], axis=1) # m × (hidden1+1)
    # Layer 2 → 3 (scaled sigmoid activation)
    z3 = a2 @ theta2.T                                           # m × hidden2
    a3 = np.concatenate([np.ones((m, 1)), sigmoid(z3)], axis=1) # m × (hidden2+1)
    # Layer 3 → output (standard sigmoid activation)
    z4 = a3 @ theta3.T                                           # m × num_labels
    h  = sigmoid_std(z4)                                         # m × num_labels
    return a1, z2, a2, z3, a3, z4, h

def unpack_params(params, input_size, hidden_size1, hidden_size2, num_labels):
    split1 = hidden_size1 * (input_size + 1)
    split2 = split1 + hidden_size2 * (hidden_size1 + 1)
    theta1 = params[:split1].reshape(hidden_size1, input_size + 1)
    theta2 = params[split1:split2].reshape(hidden_size2, hidden_size1 + 1)
    theta3 = params[split2:].reshape(num_labels, hidden_size2 + 1)
    return theta1, theta2, theta3

def cost_no_reg(params, input_size, hidden_size1, hidden_size2, num_labels, X, y):
    m = X.shape[0]
    theta1, theta2, theta3 = unpack_params(params, input_size, hidden_size1, hidden_size2, num_labels)
    _, _, _, _, _, _, h = forward_propagate(X, theta1, theta2, theta3)
    h = np.clip(h, 1e-8, 1.0 - 1e-8)
    J = -np.sum(y * np.log(h) + (1.0 - y) * np.log(1.0 - h)) / m
    return J

def cost(params, input_size, hidden_size1, hidden_size2, num_labels, X, y, learning_rate):
    m = X.shape[0]
    theta1, theta2, theta3 = unpack_params(params, input_size, hidden_size1, hidden_size2, num_labels)
    _, _, _, _, _, _, h = forward_propagate(X, theta1, theta2, theta3)
    h = np.clip(h, 1e-8, 1.0 - 1e-8)
    J = -np.sum(y * np.log(h) + (1.0 - y) * np.log(1.0 - h)) / m
    reg = (learning_rate / (2.0 * m)) * (
        np.sum(theta1[:, 1:] ** 2) +
        np.sum(theta2[:, 1:] ** 2) +
        np.sum(theta3[:, 1:] ** 2)
    )
    return J + reg

In [6]:
# Total parameter count for the 3-layer network
n_params = (hidden_size1 * (input_size + 1) +
            hidden_size2 * (hidden_size1 + 1) +
            num_labels  * (hidden_size2 + 1))
print('Total parameters:', n_params)

params = (np.random.rand(n_params) - 0.5) * 0.25
theta1, theta2, theta3 = unpack_params(params, input_size, hidden_size1, hidden_size2, num_labels)

a1, z2, a2, z3, a3, z4, h = forward_propagate(X, theta1, theta2, theta3)
print('a1, z2, a2, z3, a3, z4, h shapes:',
      a1.shape, z2.shape, a2.shape, z3.shape, a3.shape, z4.shape, h.shape)
print('Initial cost (no reg): ', cost_no_reg(params, input_size, hidden_size1, hidden_size2, num_labels, X, y_onehot))
print('Initial cost (with reg):', cost(params, input_size, hidden_size1, hidden_size2, num_labels, X, y_onehot, learning_rate))


Total parameters: 8650
a1, z2, a2, z3, a3, z4, h shapes: (5000, 401) (5000, 20) (5000, 21) (5000, 20) (5000, 21) (5000, 10) (5000, 10)
Initial cost (no reg):  6.942749351049579
Initial cost (with reg): 6.947198447109131


In [7]:
# ── Backprop without regularization
def backprop_no_reg(params, input_size, hidden_size1, hidden_size2, num_labels, X, y):
    m = X.shape[0]
    theta1, theta2, theta3 = unpack_params(params, input_size, hidden_size1, hidden_size2, num_labels)
    a1, z2, a2, z3, a3, _, h = forward_propagate(X, theta1, theta2, theta3)

    h = np.clip(h, 1e-8, 1.0 - 1e-8)
    J = -np.sum(y * np.log(h) + (1.0 - y) * np.log(1.0 - h)) / m

    # Backprop deltas
    d4 = h - y                                              # output error
    d3 = (d4 @ theta3[:, 1:]) * sigmoid_gradient(z3)       # hidden layer 2 error
    d2 = (d3 @ theta2[:, 1:]) * sigmoid_gradient(z2)       # hidden layer 1 error

    # Gradient accumulation (no regularization)
    delta1 = (d2.T @ a1) / m
    delta2 = (d3.T @ a2) / m
    delta3 = (d4.T @ a3) / m

    grad = np.concatenate([delta1.ravel(), delta2.ravel(), delta3.ravel()])
    return J, grad

# ── Backprop with L2 regularization 
def backprop(params, input_size, hidden_size1, hidden_size2, num_labels, X, y, learning_rate):
    m = X.shape[0]
    theta1, theta2, theta3 = unpack_params(params, input_size, hidden_size1, hidden_size2, num_labels)
    a1, z2, a2, z3, a3, _, h = forward_propagate(X, theta1, theta2, theta3)

    h = np.clip(h, 1e-8, 1.0 - 1e-8)
    J = -np.sum(y * np.log(h) + (1.0 - y) * np.log(1.0 - h)) / m
    J += (learning_rate / (2.0 * m)) * (
        np.sum(theta1[:, 1:] ** 2) +
        np.sum(theta2[:, 1:] ** 2) +
        np.sum(theta3[:, 1:] ** 2)
    )

    # Backprop deltas
    d4 = h - y
    d3 = (d4 @ theta3[:, 1:]) * sigmoid_gradient(z3)
    d2 = (d3 @ theta2[:, 1:]) * sigmoid_gradient(z2)

    # Gradient accumulation
    delta1 = (d2.T @ a1) / m
    delta2 = (d3.T @ a2) / m
    delta3 = (d4.T @ a3) / m

    # Add regularization gradient (skip bias column)
    delta1[:, 1:] += (learning_rate / m) * theta1[:, 1:]
    delta2[:, 1:] += (learning_rate / m) * theta2[:, 1:]
    delta3[:, 1:] += (learning_rate / m) * theta3[:, 1:]

    grad = np.concatenate([delta1.ravel(), delta2.ravel(), delta3.ravel()])
    return J, grad


In [8]:
J0_noreg, grad0_noreg = backprop_no_reg(params, input_size, hidden_size1, hidden_size2, num_labels, X, y_onehot)
print('Backprop (no reg) cost:', J0_noreg)
print('Gradient shape (no reg):', grad0_noreg.shape)

J0, grad0 = backprop(params, input_size, hidden_size1, hidden_size2, num_labels, X, y_onehot, learning_rate)
print('Backprop (with reg) cost:', J0)
print('Gradient shape (with reg):', grad0.shape)


Backprop (no reg) cost: 6.942749351049579
Gradient shape (no reg): (8650,)
Backprop (with reg) cost: 6.947198447109131
Gradient shape (with reg): (8650,)


In [9]:
result = minimize(
    fun=backprop,
    x0=params,
    args=(input_size, hidden_size1, hidden_size2, num_labels, X, y_onehot, learning_rate),
    method='TNC',
    jac=True,
    options={'maxfun': 250}
)

print('Optimization status:', result.status)
print('Optimization success:', result.success)
print('Final cost:', result.fun)


Optimization status: 3
Optimization success: False
Final cost: 0.24456047522879093


In [10]:
theta1_opt, theta2_opt, theta3_opt = unpack_params(
    result.x, input_size, hidden_size1, hidden_size2, num_labels
)
_, _, _, _, _, _, h_opt = forward_propagate(X, theta1_opt, theta2_opt, theta3_opt)

y_pred = np.argmax(h_opt, axis=1).reshape(-1, 1) + 1
accuracy_3layer = np.mean(y_pred == y) * 100.0

print(f'3-layer network accuracy = {accuracy_3layer:.2f}%')


3-layer network accuracy = 99.54%


**Part 8 — Accuracy comparison: 3-layer vs. 2-layer network**

In [11]:
# The original 2-layer template uses: standard sigmoid, 1 hidden layer of 25 units.
# Re-implement it here as a baseline for comparison (Part 8).

def sigmoid_std(z):
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_gradient_std(z):
    s = sigmoid_std(z)
    return s * (1.0 - s)

def forward_propagate_2layer(X, t1, t2):
    m = X.shape[0]
    a1 = np.concatenate([np.ones((m, 1)), X], axis=1)
    z2 = a1 @ t1.T
    a2 = np.concatenate([np.ones((m, 1)), sigmoid_std(z2)], axis=1)
    z3 = a2 @ t2.T
    h  = sigmoid_std(z3)
    return a1, z2, a2, z3, h

def backprop_2layer(params, inp, hid, nlbl, X, y, lr):
    m = X.shape[0]
    split = hid * (inp + 1)
    t1 = params[:split].reshape(hid, inp + 1)
    t2 = params[split:].reshape(nlbl, hid + 1)
    a1, z2, a2, _, h = forward_propagate_2layer(X, t1, t2)
    h = np.clip(h, 1e-8, 1 - 1e-8)
    J = -np.sum(y * np.log(h) + (1 - y) * np.log(1 - h)) / m
    J += (lr / (2 * m)) * (np.sum(t1[:, 1:] ** 2) + np.sum(t2[:, 1:] ** 2))
    d3 = h - y
    d2 = (d3 @ t2[:, 1:]) * sigmoid_gradient_std(z2)
    g1 = (d2.T @ a1) / m;  g1[:, 1:] += (lr / m) * t1[:, 1:]
    g2 = (d3.T @ a2) / m;  g2[:, 1:] += (lr / m) * t2[:, 1:]
    return J, np.concatenate([g1.ravel(), g2.ravel()])

hidden_2layer = 25
np.random.seed(301)
p0_2 = (np.random.rand(hidden_2layer * (input_size + 1) + num_labels * (hidden_2layer + 1)) - 0.5) * 0.25

res_2layer = minimize(
    fun=backprop_2layer,
    x0=p0_2,
    args=(input_size, hidden_2layer, num_labels, X, y_onehot, learning_rate),
    method='TNC', jac=True, options={'maxfun': 250}
)

split = hidden_2layer * (input_size + 1)
t1_opt = res_2layer.x[:split].reshape(hidden_2layer, input_size + 1)
t2_opt = res_2layer.x[split:].reshape(num_labels, hidden_2layer + 1)
_, _, _, _, h2 = forward_propagate_2layer(X, t1_opt, t2_opt)
accuracy_2layer = np.mean((np.argmax(h2, axis=1).reshape(-1, 1) + 1) == y) * 100.0

print(f'2-layer network accuracy (baseline) = {accuracy_2layer:.2f}%')
print(f'3-layer network accuracy             = {accuracy_3layer:.2f}%')
print(f'Difference: {accuracy_3layer - accuracy_2layer:+.2f}%')


2-layer network accuracy (baseline) = 99.40%
3-layer network accuracy             = 99.54%
Difference: +0.14%
